# PoliMillionaire Game Tests

Run live games with selected local models, one after another.

No generated-answer API is used.


## Setup

Find the repo and local model cache.


In [39]:
import gc
import os
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "polimillionaire" / "__init__.py").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the project folder.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
for path in [REPO_ROOT / "src", REPO_ROOT]:
    path_text = str(path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)

os.environ.setdefault("HF_HOME", str(REPO_ROOT / "data" / "hf_home"))
os.environ.setdefault(
    "HUGGINGFACE_HUB_CACHE", str(REPO_ROOT / "data" / "hf_home" / "hub")
)

print("Repo:", REPO_ROOT)
print("HF cache:", os.environ["HF_HOME"])


Repo: /Users/camiloa2m/Downloads/NLP/nlp-polimillionaire
HF cache: /Users/camiloa2m/Downloads/NLP/nlp-polimillionaire/data/hf_home


## Settings

Pick models here. Each selected model gets its own game run.


In [40]:
RUN_API_CHECK = True
RUN_LIVE_GAME = True
PROMPT_FOR_CREDENTIALS = False

API_URL = "http://131.175.15.22:51111/"
COMPETITION_ID = 0

MODELS_TO_RUN = [
    {"label": "Gemma 4 E2B", "model_id": "google/gemma-4-E2B-it", "run": False},
    {"label": "Gemma 4 E4B", "model_id": "google/gemma-4-E4B-it", "run": False},
    {"label": "Qwen 2.5 3B", "model_id": "Qwen/Qwen2.5-3B-Instruct", "run": False},
    {"label": "Qwen 2.5 7B", "model_id": "Qwen/Qwen2.5-7B-Instruct", "run": True},
]

SAFE_DELAY_SECONDS = 2.0
ANSWER_TIMEOUT_SECONDS = 25.0

print("API check:", RUN_API_CHECK)
print("Live game:", RUN_LIVE_GAME)
print("Competition:", COMPETITION_ID)
print("Models:")
for item in MODELS_TO_RUN:
    print("-", item["label"], item["model_id"], "run=", item["run"])


API check: True
Live game: True
Competition: 0
Models:
- Gemma 4 E2B google/gemma-4-E2B-it run= False
- Gemma 4 E4B google/gemma-4-E4B-it run= False
- Qwen 2.5 3B Qwen/Qwen2.5-3B-Instruct run= False
- Qwen 2.5 7B Qwen/Qwen2.5-7B-Instruct run= True


## Login

Use environment variables, Colab secrets, or prompt mode.


In [41]:
import getpass

from dotenv import load_dotenv

from millionaire_client import AuthenticationError, MillionaireClient

# Load environment variables from .env file
load_dotenv()

USERNAME = os.environ.get("POLIMILLIONAIRE_USERNAME")
PASSWORD = os.environ.get("POLIMILLIONAIRE_PASSWORD")

try:
    from google.colab import userdata

    USERNAME = USERNAME or userdata.get("POLIMILLIONAIRE_USERNAME")
    PASSWORD = PASSWORD or userdata.get("POLIMILLIONAIRE_PASSWORD")
except Exception:
    pass

if PROMPT_FOR_CREDENTIALS and not USERNAME:
    USERNAME = input("PoliMillionaire username: ").strip()
if PROMPT_FOR_CREDENTIALS and not PASSWORD:
    PASSWORD = getpass.getpass("PoliMillionaire password: ")

print("Username configured:", bool(USERNAME))
print("Password configured:", bool(PASSWORD))


Username configured: True
Password configured: True


## Competitions

Turn on `RUN_API_CHECK` to list games.


In [42]:
client = None

if RUN_API_CHECK:
    if not USERNAME or not PASSWORD:
        raise RuntimeError("Set credentials first.")
    try:
        client = MillionaireClient(API_URL)
        user = client.login(USERNAME, PASSWORD)
        print("Logged in as", user.username)
        for competition in client.competitions.list_all():
            print(competition.id, competition.name, competition.max_levels)
    except AuthenticationError as exc:
        client = None
        print("Login failed:", exc)
else:
    print("API check skipped.")


Logged in as promptPotamo
0 Entertainment 15
1 Ancient History and Politics 15
2 Science and Nature 15
3 Maths 15
4 Philosophy and Psychology 15
5 News 15


## Run Selected Models

Each model warms up, plays if enabled, then unloads.


In [43]:
# !pip install  -qU colab-xterm
# %load_ext colabxterm
# # It may ask you to restart, accept and run this cell again

### For Colab: Run the following commands on the terminal below:
To spawn the server deamon please run the cell with ```%xterm```. It will create a concurrent shell where we can paste the following commands but, at the same time, also run other cells!

```sudo apt-get install zstd```

```curl -fsSL https://ollama.com/install.sh | sh```

```ollama serve & ollama pull llama3```

In [44]:
# %xterm

In [45]:
import logging
import sys
from typing import Any

from src.polimillionaire.types import AnswerOption, Question

root = logging.getLogger()
if not any(isinstance(h, logging.StreamHandler) for h in root.handlers):
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(
        logging.Formatter("%(asctime)s %(levelname)s %(name)s: %(message)s")
    )
    root.addHandler(handler)
root.setLevel(logging.INFO)


In [ ]:
"""
RAG Web Retrieval Pipeline
"""

from __future__ import annotations

import asyncio
import hashlib
import logging
import time
from dataclasses import dataclass
from typing import Optional

import httpx
import nest_asyncio
import numpy as np
import trafilatura
from ddgs import DDGS
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field
from sentence_transformers import CrossEncoder, SentenceTransformer

# -- allow nested event loop for Jupyter / IPython compatibility -------------
nest_asyncio.apply()

# -- optional: swap for newspaper3k / BeautifulSoup as fallback --------------
try:
    from newspaper import Article as NewspaperArticle

    HAS_NEWSPAPER = True
except ImportError:
    HAS_NEWSPAPER = False


# -- logging setup -----------------------------------------------------------
log = logging.getLogger(__name__)

# ===============================================================================
# MODULE-LEVEL SINGLETONS 
# ===============================================================================

# One splitter instance reused across all URL fetches
_SPLITTER = RecursiveCharacterTextSplitter(
    chunk_size=450,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " ", ""],
)

# Cross-encoder loaded at import time, not on first query
log.info("Loading cross-encoder model at startup …")
_CROSS_ENCODER = CrossEncoder("BAAI/bge-reranker-base")

# Lightweight bi-encoder runs on-device
log.info("Loading dense encoder model at startup …")
_EMBED_MODEL = SentenceTransformer("BAAI/bge-small-en-v1.5")


# ===============================================================================
# CONFIG
# ===============================================================================


@dataclass
class RetrievalConfig:
    # Search
    max_ddg_results: int = 7
    timeout_ddg: int = 10
    num_extra_queries: int = 1

    # Fetching
    fetch_timeout: float = 8.0
    max_concurrent_fetches: int = 6
    max_text_chars: int = 20_000
    total_fetch_budget: float = 9.0

    # Chunking
    chunk_size: int = 450
    chunk_overlap: int = 80

    # Retrieval
    bm25_top_k: int = 10
    dense_top_k: int = 10
    rrf_k: int = 60
    diversity_max_per_url: int = 2

    # Reranking
    final_top_k: int = 3

    # LLM (query expansion + answering)
    llm_model: str = "qwen2.5:7b-instruct-q4_K_M"


# ===============================================================================
# QUERY EXPANSION
# ===============================================================================

_EXPANSION_PROMPT = """\
You are a web search-query optimizer. Given the user query below, generate {n} \
alternative search queries that cover different phrasings, synonyms, or \
sub-aspects of the same information need.

Rules:
- Each alternative must be semantically distinct from the others.
- Keep each query concise (less than 12 words).
- Output ONLY the queries, one per line, no numbering, no explanation.

User query: {query}
"""


def expand_query(query: str, cfg: RetrievalConfig) -> list[str]:
    """Return original query + N LLM-generated variants."""
    if cfg.num_extra_queries == 0:
        return [query]

    llm = ChatOllama(model=cfg.llm_model, temperature=0)

    prompt = _EXPANSION_PROMPT.format(n=cfg.num_extra_queries, query=query)

    try:
        response = llm.invoke(prompt)
        variants = [
            line.strip()
            for line in response.content.strip().splitlines()
            if line.strip()
        ][: cfg.num_extra_queries]
    except Exception as exc:
        log.warning("Query expansion failed, falling back to original: %s", exc)
        variants = []

    all_queries = [query] + variants
    log.info("Expanded queries: %s", all_queries)
    return all_queries


# ===============================================================================
# SEARCH & DEDUPLICATION
# ===============================================================================


def ddg_search(query: str, cfg: RetrievalConfig) -> list[dict]:
    with DDGS(timeout=cfg.timeout_ddg) as ddgs:
        return list(
            ddgs.text(
                query,
                max_results=cfg.max_ddg_results,
                backend="yandex, mojeek, startpage, yahoo",
            )
        )


def search_all_queries(queries: list[str], cfg: RetrievalConfig) -> list[dict]:
    """Search every expanded query and merge, deduplicating by URL."""
    seen_urls: set[str] = set()
    merged: list[dict] = []

    for q in queries:
        try:
            results = ddg_search(q, cfg)
        except Exception as exc:
            log.warning("DDG search failed for %r: %s", q, exc)
            continue

        for r in results:
            url = (r.get("href") or "").strip().lower().rstrip("/")
            if not url or url in seen_urls:
                continue
            seen_urls.add(url)
            merged.append(r)

    log.info("Total unique URLs after merging expansions: %d", len(merged))
    return merged


# ===============================================================================
# ASYNC FETCHING
# ===============================================================================

_HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; RAGBot/1.0)"}

_BLOCKED_EXTENSIONS = {".pdf", ".doc", ".docx", ".ppt", ".pptx", ".xls", ".xlsx"}
_BLOCKED_DOMAINS = {
    "twitter.com",
    "x.com",
    "instagram.com",
    "facebook.com",
    "tiktok.com",
    "reddit.com",
}


def _is_fetchable(url: str) -> bool:
    from urllib.parse import urlparse

    parsed = urlparse(url.lower())
    if any(parsed.netloc.endswith(d) for d in _BLOCKED_DOMAINS):
        return False
    if any(parsed.path.endswith(ext) for ext in _BLOCKED_EXTENSIONS):
        return False
    return True


def _extract_text(html: str) -> str:
    """Trafilatura -> newspaper3k fallback."""
    text = trafilatura.extract(
        html,
        include_comments=False,
        include_tables=False,
        favor_recall=True,
    )
    if text and len(text) >= 200:
        return text

    if HAS_NEWSPAPER:
        try:
            article = NewspaperArticle(url="")
            article.set_html(html)
            article.parse()
            if article.text and len(article.text) >= 200:
                return article.text
        except Exception:
            pass

    return ""


async def _fetch_one(
    client: httpx.AsyncClient,
    semaphore: asyncio.Semaphore,
    result: dict,
    cfg: RetrievalConfig,
) -> list[Document]:
    """Fetch a single URL and return chunked Documents."""
    url: str = result.get("href", "")
    title: str = result.get("title", "No title")
    snippet: str = result.get("body", "")

    if not url or not _is_fetchable(url):
        return (
            [Document(page_content=snippet, metadata={"title": title, "url": url, "snippet": snippet, "chunk_id": 0})]
            if snippet
            else []
        )

    async with semaphore:
        try:
            resp = await client.get(url, headers=_HEADERS, timeout=cfg.fetch_timeout)
            resp.raise_for_status()
            html = resp.text
        except Exception as exc:
            log.debug("Fetch failed for %s: %s", url, exc)
            return (
                [Document(page_content=snippet, metadata={"title": title, "url": url, "snippet": snippet, "chunk_id": 0})]
                if snippet
                else []
            )

    text = _extract_text(html)
    if not text:
        text = snippet

    text = " ".join(text.split())[: cfg.max_text_chars]

    chunks = _SPLITTER.split_text(text)

    return [
        Document(
            page_content=chunk,
            metadata={"title": title, "url": url, "snippet": snippet, "chunk_id": i},
        )
        for i, chunk in enumerate(chunks)
    ]


async def fetch_all_async(results: list[dict], cfg: RetrievalConfig) -> list[Document]:
    """Fetch all URLs concurrently within a hard wall-clock budget."""
    semaphore = asyncio.Semaphore(cfg.max_concurrent_fetches)

    async with httpx.AsyncClient(follow_redirects=True) as client:
        tasks = [
            asyncio.ensure_future(_fetch_one(client, semaphore, r, cfg))
            for r in results
        ]

        # Total_fetch_budget caps the entire fetch phase regardless of
        # how many slow servers are in the result set
        done, pending = await asyncio.wait(tasks, timeout=cfg.total_fetch_budget)

        if pending:
            log.info("Cancelling %d slow fetches (budget %.1fs exceeded)", len(pending), cfg.total_fetch_budget)
            for t in pending:
                t.cancel()

    docs: list[Document] = []
    for fut in done:
        try:
            batch = fut.result()
            if isinstance(batch, Exception):
                log.warning("Fetch task raised: %s", batch)
                continue
            docs.extend(batch)
        except Exception as exc:
            log.warning("Fetch task raised: %s", exc)

    log.info("Total chunks after async fetch: %d", len(docs))
    return docs


def fetch_all(results: list[dict], cfg: RetrievalConfig) -> list[Document]:
    """Sync entry-point compatible with Jupyter + scripts."""
    loop = asyncio.get_event_loop()
    if loop.is_running():
        return loop.run_until_complete(fetch_all_async(results, cfg))
    else:
        return asyncio.run(fetch_all_async(results, cfg))


# ===============================================================================
# DIVERSITY FILTER
# ===============================================================================


def diversity_filter(docs: list[Document], max_per_url: int = 2) -> list[Document]:
    seen: dict[str, int] = {}
    filtered = []
    for doc in docs:
        url = doc.metadata.get("url", "")
        if seen.get(url, 0) >= max_per_url:
            continue
        filtered.append(doc)
        seen[url] = seen.get(url, 0) + 1
    return filtered


# ===============================================================================
# HYBRID RETRIEVAL: BM25 + DENSE + RRF
# ===============================================================================


def _rrf_score(rank: int, k: int = 60) -> float:
    return 1.0 / (k + rank)


def _dense_retrieve(query: str, docs: list[Document], top_k: int) -> list[Document]:
    """
    Encode with the module-level SentenceTransformer directly on GPU.
    Replaces FAISS.from_documents + OllamaEmbeddings which rebuilt the full
    index on every query via a network round-trip to the Ollama server.
    """
    if not docs:
        return []

    texts = [d.page_content for d in docs]

    doc_embs = _EMBED_MODEL.encode(
        texts,
        batch_size=64,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    q_emb = _EMBED_MODEL.encode(query, normalize_embeddings=True)

    scores = np.dot(doc_embs, q_emb)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [docs[i] for i in top_indices]


def hybrid_retrieve_rrf(
    query: str,
    docs: list[Document],
    cfg: RetrievalConfig,
) -> list[Document]:
    """
    1. BM25 retrieval   -> ranked list A
    2. Dense retrieval  -> ranked list B
    3. Fuse with RRF    -> final merged ranking
    """
    if not docs:
        return []

    # BM25
    bm25_retriever = BM25Retriever.from_documents(docs, k=cfg.bm25_top_k)
    bm25_results: list[Document] = bm25_retriever.invoke(query)

    # Dense
    dense_results = _dense_retrieve(query, docs, cfg.dense_top_k)

    # RRF Fusion
    def _doc_key(doc: Document) -> str:
        return "{}::{}".format(
            doc.metadata.get("url", ""),
            doc.metadata.get("chunk_id", hashlib.md5(doc.page_content.encode()).hexdigest()[:8]),
        )

    scores: dict[str, float] = {}
    doc_map: dict[str, Document] = {}

    for rank, doc in enumerate(bm25_results):
        key = _doc_key(doc)
        scores[key] = scores.get(key, 0.0) + _rrf_score(rank, cfg.rrf_k)
        doc_map[key] = doc

    for rank, doc in enumerate(dense_results):
        key = _doc_key(doc)
        scores[key] = scores.get(key, 0.0) + _rrf_score(rank, cfg.rrf_k)
        doc_map[key] = doc

    ranked_keys = sorted(scores, key=scores.__getitem__, reverse=True)
    fused = []
    for k in ranked_keys:
        doc = doc_map[k]
        doc.metadata["rrf_score"] = scores[k]
        fused.append(doc)

    log.info(
        "RRF fusion: BM25=%d, dense=%d -> merged=%d",
        len(bm25_results),
        len(dense_results),
        len(fused),
    )
    return fused


# ===============================================================================
# CROSS-ENCODER RERANKING
# ===============================================================================


def cross_encoder_rerank(
    query: str,
    docs: list[Document],
    cfg: RetrievalConfig,
) -> list[Document]:
    if not docs:
        return []

    pairs = [(query, doc.page_content) for doc in docs]

    # Use the module-level singleton loaded at startup
    try:
        scores = _CROSS_ENCODER.predict(pairs)
    except Exception as exc:
        log.warning("Cross-encoder failed, returning unranked docs: %s", exc)
        return docs[: cfg.final_top_k]

    ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
    top_docs = [doc for _, doc in ranked[: cfg.final_top_k]]

    log.info("Cross-encoder reranked %d -> %d docs", len(docs), len(top_docs))
    return top_docs


# ===============================================================================
# FORMATTING
# ===============================================================================


def format_evidence(docs: list[Document]) -> str:
    if not docs:
        return "No results found."

    blocks = []
    for i, doc in enumerate(docs, 1):
        blocks.append(
            f"[Chunk {i}]\n"
            f"TITLE: {doc.metadata.get('title')}\n"
            f"URL:   {doc.metadata.get('url')}\n"
            f"CONTENT:  {doc.page_content}\n"
            f"SNIPPET: {doc.metadata.get('snippet')}"
        )
    return "\n\n".join(blocks)


# ===============================================================================
# RETRIEVAL PIPELINE
# ===============================================================================


def retrieve(query: str, cfg: Optional[RetrievalConfig] = None) -> str:
    """
    Full pipeline:
      query -> expand -> search -> async fetch
            -> hybrid RRF retrieval -> diversity filter  ← Fix #1: moved before rerank
            -> cross-encoder rerank -> format
    """
    cfg = cfg or RetrievalConfig()
    t0 = time.perf_counter()

    # 1. Query expansion
    queries = expand_query(query, cfg)

    # 2. Search all variants, deduplicate by URL
    raw_results = search_all_queries(queries, cfg)
    if not raw_results:
        return "No results found."

    # 3. Async fetch + chunk (with total budget cap)
    docs = fetch_all(raw_results, cfg)

    # 4. Hybrid BM25 + dense + RRF
    fused_docs = hybrid_retrieve_rrf(query, docs, cfg)
    if not fused_docs:
        return "No results found."

    #  diversity filter runs on the full RRF-ranked set BEFORE the
    # cross-encoder caps results to final_top_k; previously it ran after,
    # meaning all final_top_k docs could come from one URL and be filtered out.
    diverse_docs = diversity_filter(fused_docs, max_per_url=cfg.diversity_max_per_url)

    # 5. Cross-encoder rerank (now receives a diverse candidate set)
    top_docs = cross_encoder_rerank(query, diverse_docs, cfg)

    elapsed = time.perf_counter() - t0
    log.info("retrieve() completed in %.2fs, returning %d chunks", elapsed, len(top_docs))

    return format_evidence(top_docs)


# ===============================================================================
# STRUCTURED OUTPUT SCHEMA
# ===============================================================================


class TriviaAnswer(BaseModel):
    answer: int = Field(description="Option ID")
    confidence: float = Field(
        ge=0.00,
        le=1.00,
        description="Confidence score decimal number between 0.00 and 1.00, 2 decimals precision",
    )
    evidence: str = Field(description="Short evidence-based justification")


# ===============================================================================
# ANSWER — AUGMENTED GENERATION WITH RETRIEVAL AND STRUCTURED OUTPUT
# ===============================================================================

SYSTEM_PROMPT = """
You are a careful multiple-choice trivia solver.

Rules:
1. Use evidence primarily.
2. Use internal knowledge only to supplement evidence.
3. Compare ALL options carefully.
4. Select EXACTLY one option ID.
5. Lower confidence if evidence is weak.
"""

REASONING_PROMPT = """
QUESTION:
{question}

EVIDENCE:
{evidence}

Confidence score is a decimal number between 0.00 (lowest) and 1.00 (highest).
Return the best option.
"""


def _build_mcq(question) -> str:
    """Format question + options as text for the prompt."""
    options = "\n".join(f"{option.id}) {option.text}" for option in question.options)
    return f"{question.text}\n{options}"


def answer(question, cfg: Optional[RetrievalConfig] = None) -> str:
    """Return the best answer for the given question and evidence."""
    cfg = cfg or RetrievalConfig()

    llm = ChatOllama(model=cfg.llm_model, temperature=0)

    evidence = retrieve(question.text, cfg)
    log.info("Evidence retrieved:\n%s", evidence)

    mcq_text = _build_mcq(question)

    reasoning_prompt = ChatPromptTemplate.from_messages(
        [
            ("system", SYSTEM_PROMPT),
            ("human", REASONING_PROMPT),
        ]
    )

    reasoning_llm = llm.with_structured_output(TriviaAnswer)
    reasoning_chain = reasoning_prompt | reasoning_llm

    response = reasoning_chain.invoke({"question": mcq_text, "evidence": evidence})

    return response.model_dump_json(indent=2)


if __name__ == "__main__":
    from dataclasses import dataclass as _dc

    @_dc
    class AnswerOption:
        id: int
        text: str

    @_dc
    class Question:
        id: int
        text: str
        options: list

    recent_question = Question(
        id=3,
        text="Who won the UEFA Champions League in 2025?",
        options=[
            AnswerOption(0, "Real Madrid"),
            AnswerOption(1, "Manchester City"),
            AnswerOption(2, "PSG"),
            AnswerOption(3, "Bayern Munich"),
        ],
    )

    answer_text = answer(recent_question)
    print("LLM answer:", answer_text)

2026-05-28 10:21:35,202 INFO __main__: Loading cross-encoder model at startup …
2026-05-28 10:21:35,220 INFO sentence_transformers.base.model: No device provided, using mps
2026-05-28 10:21:35,415 INFO httpx: HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/modules.json "HTTP/1.1 404 Not Found"
2026-05-28 10:21:35,419 INFO sentence_transformers.base.model: No modules.json found for BAAI/bge-reranker-base, initializing a new CrossEncoder model.
2026-05-28 10:21:35,547 INFO httpx: HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-base "HTTP/1.1 200 OK"
2026-05-28 10:21:35,678 INFO httpx: HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-28 10:21:35,688 INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-base/2cfc18c9415c912f9d8155881c133215df768a70/config.json "HTTP/1.1 200 OK"
2026-05-28 10:21:35,831 INFO httpx: H

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7198.83it/s]

2026-05-28 10:21:36,260 INFO httpx: HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"


2026-05-28 10:21:36,387 INFO httpx: HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"


KeyboardInterrupt: 

In [ ]:
type(answer_text)

__main__.TriviaAnswer

In [ ]:
from abc import ABC, abstractmethod
from typing import Any, Protocol

from src.polimillionaire.strategies import parse_answer_prediction


class BaseStrategy(ABC):
    name = "base"

    @abstractmethod
    def answer(self, question: Question) -> AnswerPrediction:
        raise NotImplementedError


class LocalLLM(Protocol):
    model_name: str

    def generate(self, prompt: str, **kwargs: object) -> str: ...


class OllamaStrategy(BaseStrategy):
    name = "Ollama-Gwen2.5-7b"

    def __init__(self):
        self.llm = ChatOllama(model="gwen2.5:7b", temperature=0)

    def answer(self, question: Question) -> AnswerPrediction:
        prediction = ask_question(question)

        # raw_text = self.llm.generate(build_prompt(question))
        prediction = parse_answer_prediction(
            str(prediction), question, strategy_name=self.name
        )
        prediction.metadata["strategy"] = self.name
        prediction.metadata["model_name"] = getattr(self.llm, "model_name", "unknown")
        prediction.metadata["device"] = getattr(self.llm, "device_summary", "unknown")
        return prediction

In [ ]:
from datetime import datetime, timezone

from polimillionaire.runner import GameRunner, RunLogger
from polimillionaire.strategies import GemmaLLMConfig, GemmaStrategy
from polimillionaire.types import AnswerOption, Question

warmup_question = Question(
    id=1,
    text="What is 2 + 2?",
    options=[
        AnswerOption(0, "3"),
        AnswerOption(1, "4"),
        AnswerOption(2, "5"),
        AnswerOption(3, "22"),
    ],
)


def clear_model_memory():
    gc.collect()
    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        if (
            hasattr(torch, "mps")
            and hasattr(torch.backends, "mps")
            and torch.backends.mps.is_available()
        ):
            torch.mps.empty_cache()
    except Exception as exc:
        print("Cleanup warning:", exc)


def make_strategy(model_id: str) -> GemmaStrategy:
    config = GemmaLLMConfig(
        model_id=model_id,
        inference_backend="auto_model",
        device_map="auto",
        dtype="auto",
        max_new_tokens=8,
        temperature=0.0,
        do_sample=False,
        num_beams=1,
        seed=42,
        generation_max_time_seconds=18.0,
        timeout_seconds=120.0,
    )
    return GemmaStrategy(model_config=config)


def clean_name(label: str) -> str:
    return "_".join(label.lower().split())


if RUN_LIVE_GAME and (not USERNAME or not PASSWORD):
    raise RuntimeError("Set credentials first.")
if RUN_LIVE_GAME and client is None:
    client = MillionaireClient(API_URL)
    client.login(USERNAME, PASSWORD)

run_results = []
for item in MODELS_TO_RUN:
    if not item.get("run", False):
        print("Skipped", item["label"])
        continue

    strategy = None
    try:
        print()
        print("Model:", item["label"])
        # strategy = make_strategy(item["model_id"])
        strategy = OllamaStrategy()

        warmup = strategy.answer(warmup_question)

        print("warmup option:", warmup.option_id, warmup.answer_text)
        print("fallback:", warmup.metadata.get("fallback"))
        print("device:", warmup.metadata.get("device"))
        if warmup.metadata.get("fallback"):
            raise RuntimeError(f"Warm-up failed for {item['label']}.")

        result = {
            "label": item["label"],
            "model_id": item["model_id"],
            "warmup_option": warmup.option_id,
        }

        if RUN_LIVE_GAME:
            run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
            log_path = (
                REPO_ROOT
                / "results"
                / "runs"
                / f"{clean_name(item['label'])}_{run_id}.jsonl"
            )
            runner = GameRunner(
                client=client,
                safe_delay_seconds=SAFE_DELAY_SECONDS,
                answer_timeout_seconds=ANSWER_TIMEOUT_SECONDS,
                logger=RunLogger(log_path),
            )
            game = runner.play(COMPETITION_ID, strategy)
            result.update(
                {
                    "current_level": game.current_level,
                    "earned_amount": game.earned_amount,
                    "log_path": str(log_path),
                }
            )
            print("Reached level:", game.current_level)
            print("Earned amount:", game.earned_amount)
            print("Log path:", log_path)
        else:
            print("Live game skipped for", item["label"])

        run_results.append(result)
    finally:
        del strategy
        clear_model_memory()
        print("Cleared model memory after", item["label"])

run_results


## Results

Read recent game logs.


In [ ]:
from polimillionaire.runner import load_jsonl, summarize_attempts

run_logs = sorted(
    (REPO_ROOT / "results" / "runs").glob("*.jsonl"),
    key=lambda path: path.stat().st_mtime,
)

for path in run_logs[-5:]:
    print(path.name)

if run_logs:
    rows = load_jsonl(run_logs[-1])
    print("latest:", run_logs[-1].name)
    print(summarize_attempts(rows))
else:
    print("No logs found.")


## Done

Use `run_results` and the JSONL logs for comparison.
